# Probability distributions and conditional laws

This problem book exercises Convexity's second mathematical family. All inputs
are mathematical parameters or declared finite outcomes. There is no market data,
model fitting or forecasting. NumPy/SciPy perform the numerical work behind a
common law interface; optional PyTorch laws preserve gradients.

Run all cells in an environment with `pip install -e '.[notebooks,autodiff]'`.
PyTorch is optional; the final example skips explicitly when it is absent.
See [the catalog and contracts](../docs/distributions.md) for supported
operations, shape conventions and numerical limitations.

In [ ]:
from pathlib import Path
import sys

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "convexity" / "math").is_dir())
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import numpy as np
import matplotlib.pyplot as plt
from numpy.testing import assert_allclose
from convexity.math import distributions as dist

rng = np.random.default_rng(42)
plt.rcParams.update({"figure.figsize": (9, 3.5), "axes.grid": True, "grid.alpha": .2})

## 1. Scalar laws: density, tails and moments

A Gaussian's `scale` is its standard deviation. A Student-t's `scale` is a
different parameter: for 5 degrees of freedom, scale $\sqrt{3/5}$ gives unit
variance. Compare laws with the same mean and variance but different tails.
`survival(x)` evaluates $P(X>x)$ directly. It remains useful when a CDF rounds to one.

In [ ]:
gaussian = dist.normal()
student = dist.student_t(5, scale=np.sqrt(3/5))
assert_allclose(gaussian.variance(), student.variance())
x = np.linspace(-5, 5, 700)
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
for law, label in [(gaussian, "Gaussian"), (student, "Student-t, df=5")]:
    axes[0].plot(x, law.density(x), label=label)
    positive = x[x >= 0]
    axes[1].semilogy(positive, law.survival(positive), label=label)
axes[0].set(xlabel="x", ylabel="Density", title="Equal variance, different shape")
axes[1].set(xlabel="x", ylabel="P(X > x)", title="Upper tails")
axes[0].legend(); axes[1].legend()
plt.tight_layout(); plt.show()
assert_allclose(dist.normal(2, 3).expect(lambda x: x*x), 13.)
print("Gaussian P(X > 10):", float(gaussian.survival(10)))
print("Gaussian 5/50/95% quantiles:", gaussian.quantile([.05, .5, .95]))

## 2. Discrete laws and inverse CDFs

The same `density` method returns a **probability mass** for discrete laws.
CDFs include equality; survival probabilities exclude it. Quantiles choose an
actual supported outcome. Bernoulli, binomial and Poisson models have no fitting
step here: their parameters are supplied directly.

In [ ]:
counts = dist.poisson(3)
successes = dist.binomial(10, .3)
k = np.arange(13)
fig, ax = plt.subplots()
ax.bar(k-.18, counts.density(k), width=.36, label="Poisson(rate=3)")
ax.bar(k+.18, successes.density(k), width=.36, label="Binomial(n=10, p=.3)")
ax.set(xlabel="Outcome", ylabel="Probability mass", title="Same mean; different count laws")
ax.legend(); plt.tight_layout(); plt.show()
assert_allclose(counts.mean(), successes.mean())
assert_allclose(counts.cdf(2) + counts.survival(2), 1)
print("P(Poisson(3) <= 2):", float(counts.cdf(2)))
print("Binomial quantiles:", successes.quantile([0, .25, .5, .75, 1]))
print("Bernoulli P(X=1):", float(dist.bernoulli(.3).density(1)))

## 3. Empirical laws are exact finite probability measures

Weights are relative masses. Repeated atoms add mass; zero-mass values do not
affect support or moments. There is no quantile interpolation or assumed tail.
Vector empirical laws preserve joint outcomes and support exact conditioning.

In [ ]:
atoms = dist.empirical([0, 10, 10, 999], weights=[1, 1, 2, 0])
assert_allclose(atoms.density([0, 10, 999]), [.25, .75, 0])
assert_allclose(atoms.quantile([0, .25, .26, 1]), [0, 0, 10, 10])
assert_allclose(atoms.expect(lambda x: [x, x*x]), [7.5, 75])
print("Atom masses:", atoms.density([0, 10]))
print("P(X > 0):", float(atoms.probability(lambda x: x > 0)))

joint_atoms = dist.empirical([[0, 1], [0, 3], [1, 4]], [1, 3, 4])
conditional_atoms = joint_atoms.condition([0], [0])
assert_allclose(conditional_atoms.mean(), [2.5])
print("E[X1 | X0=0]:", conditional_atoms.mean())
print("Exact law of the sum: mean =", float(joint_atoms.map(lambda x: x.sum()).mean()))

## 4. Mixtures and independent products

A mixture chooses a component and then draws from it. Its variance includes
both within-component variation and separation of component means. The component
laws only need to implement the public interface. An independent product instead
draws every component and produces a vector event.

In [ ]:
mixed = dist.mixture([dist.normal(-2, 1), dist.normal(4, 2)], weights=[1, 3])
assert_allclose(mixed.mean(), 2.5)
assert_allclose(mixed.variance(), 10)
p = np.array([.1, .5, .9])
q = mixed.quantile(p)
assert_allclose(mixed.cdf(q), p, atol=1e-13)
x = np.linspace(-6, 11, 700)
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].plot(x, mixed.density(x), label="Mixture density")
axes[0].hist(mixed.sample(8000, rng=rng), bins=60, density=True, alpha=.25, label="Draws from the law")
axes[0].set(xlabel="x", ylabel="Density", title="Mixture and reproducible draws")
axes[0].legend()
axes[1].plot(x, mixed.cdf(x)); axes[1].scatter(q, p)
axes[1].set(xlabel="x", ylabel="P(X <= x)", title="Inverse CDF checks")
plt.tight_layout(); plt.show()
product = dist.independent([dist.normal(1, 2), dist.student_t(5)])
print("Product sample shape:", product.sample((2, 3), rng=rng).shape)
print("Product covariance:\n", product.covariance())

## 5. Vector Gaussian and Student-t conditional laws

For a joint Gaussian with unit variances and correlation 0.5,
$X_0\mid X_1=2$ has mean 1 and variance 0.75. Conditioning an elliptical
Student-t changes both degrees of freedom and shape. The density identity
$f(x_0,x_1)=f(x_0\mid x_1)f(x_1)$ checks the returned law independently.

In [ ]:
joint_gaussian = dist.multivariate_normal([0, 0], [[1, .5], [.5, 1]])
joint_student = dist.multivariate_student_t(5, [0, 0], [[1, .5], [.5, 1]])
gaussian_given = joint_gaussian.condition([1], [2])
student_given = joint_student.condition([1], [2])
assert_allclose(gaussian_given.mean(), [1])
assert_allclose(gaussian_given.covariance(), [[.75]])
assert_allclose(student_given.covariance(), [[1.6875]])
for joint, conditional in [(joint_gaussian, gaussian_given), (joint_student, student_given)]:
    assert_allclose(conditional.log_density([1]) + joint.marginal([1]).log_density([2]), joint.log_density([1, 2]))
x = np.linspace(-4, 6, 600)
fig, ax = plt.subplots()
ax.plot(x, joint_gaussian.marginal([0]).density(x[:, None]), label="Gaussian marginal")
ax.plot(x, gaussian_given.density(x[:, None]), label="Gaussian given X1=2")
ax.plot(x, student_given.density(x[:, None]), label="Student-t given X1=2")
ax.set(xlabel="X0", ylabel="Density", title="Conditioning returns a new probability law")
ax.legend(); plt.tight_layout(); plt.show()

mixture_joint = dist.mixture([dist.multivariate_normal([0, 0], np.eye(2)),
                             dist.multivariate_normal([2, 4], np.eye(2))], [1, 3])
posterior_weight = 3*np.exp(-2)/(1+3*np.exp(-2))
assert_allclose(mixture_joint.condition([0], [0]).mean(), [4*posterior_weight])
print("Mixture prior second-component weight: 0.75")
print("Weight after observing X0=0:", posterior_weight)

## 6. Transformations and change of variables

Exponentiating a Gaussian produces a lognormal law. Convexity's general
transform uses a supplied inverse and log absolute derivative. Compare it with
the direct SciPy-backed lognormal. A negative affine scale reverses CDF and
quantile orientation; its moments follow algebraically.

In [ ]:
positive = dist.exponential(dist.normal(.2, .7))
reference = dist.lognormal(.2, .7)
x = np.geomspace(.01, 20, 300)
assert_allclose(positive.log_density(x), reference.log_density(x), atol=1e-12)
assert_allclose(positive.cdf(x), reference.cdf(x), atol=1e-14)
assert_allclose(positive.expect(lambda x: x), reference.mean(), rtol=1e-8)
assert_allclose(positive.density([-1, 0]), [0, 0])
reversed_law = dist.affine(dist.normal(2, 3), shift=4, scale=-2)
assert_allclose(reversed_law.mean(), 0)
assert_allclose(reversed_law.variance(), 36)
print("E[exp(Normal(.2,.7))]:", float(positive.expect(lambda x: x)))
print("Negative affine 10/50/90% quantiles:", reversed_law.quantile([.1, .5, .9]))

## 7. Extend the interface with a custom law

This point mass implements only sampling and log mass. Those capabilities are
enough to compose it with another discrete law. Unsupported operations remain
explicit instead of silently choosing an approximation.

In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True)
class PointMass:
    event_shape = ()
    measure: dist.Measure = "discrete"
    capabilities = frozenset({"log_density", "sample"})

    def evaluate(self, operation, *args, **kwargs):
        if operation == "sample":
            return np.full(args[0], 3.)
        return np.where(args[0] == 3., 0., -np.inf)

custom = dist.Distribution(PointMass())
combined = dist.mixture([custom, dist.empirical([0., 1.])])
assert_allclose(combined.density([0, 1, 3]), [.25, .25, .5])
print("Custom mixture masses:", combined.density([0, 1, 3]))
print("Declared operations:", sorted(combined.capabilities))

## 8. Optional differentiable tensor laws

The tensor namespace preserves PyTorch tensors, dtype, device and gradient
graphs. `rsample` is a pathwise sample only for laws that support it. Ordinary
discrete sampling does not gain a pathwise gradient. RNG control uses PyTorch's
native seed/context, separate from NumPy generators.

In [ ]:
try:
    import torch
except ModuleNotFoundError as exc:
    if exc.name != "torch":
        raise
    print("Skipped optional tensor example: install convexity[autodiff].")
else:
    loc = torch.tensor(2., dtype=torch.float64, requires_grad=True)
    tensor_law = dist.tensor.normal(loc, torch.tensor(3., dtype=torch.float64))
    log_probability = tensor_law.log_density(torch.tensor(1., dtype=torch.float64))
    gradient = torch.autograd.grad(log_probability, loc)[0]
    assert_allclose(gradient, -1/9)
    with torch.random.fork_rng():
        torch.manual_seed(42)
        draws = tensor_law.rsample(100)
    pathwise_gradient = torch.autograd.grad(draws.mean(), loc)[0]
    assert_allclose(pathwise_gradient, 1.)
    print("d log_density(1) / d loc:", float(gradient))
    print("d mean(pathwise draws) / d loc:", float(pathwise_gradient))
    print("Discrete pathwise sampling supported:", "rsample" in dist.tensor.poisson(3.).capabilities)

print("All probability identities and shape checks passed.")